## 1. Setup
Install dependencies and confirm GPU availability.

In [2]:
!pip install -q jiwer
# transformers, torch, and datasets are pre-installed on Kaggle, but this keeps them current
!pip install -q -U transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 96.1 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 66.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 105.4 MB/s eta 0:00:0000:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not in

In [3]:
import torch
print(torch.cuda.is_available())  # should print True if GPU is on

True


## 2. Data
Download the Basque Common Voice corpus (v27) via the Mozilla Data Collective API, then extract only the test split we need (test.tsv + matching audio clips) rather than the full 14GB archive.

In [5]:
import requests
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("MOZILLA")

response = requests.post(
    "https://mozilladatacollective.com/api/datasets/cmu5xc07l00e6o10740qugb6a/download",
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
)
download_url = response.json()["downloadUrl"]

with requests.get(download_url, stream=True) as r:
    with open("basque-common-voice.tar.gz", "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

print("Download complete")

Download complete


In [6]:
import tarfile

with tarfile.open("basque-common-voice.tar.gz", "r:gz") as tar:
    names = tar.getnames()
    print(names[:20])  # first 20 entries, to see the folder structure

['cv-corpus-27.0-2026-09-11/eu/README.md', 'cv-corpus-27.0-2026-09-11/eu/clip_durations.tsv', 'cv-corpus-27.0-2026-09-11/eu/clips', 'cv-corpus-27.0-2026-09-11/eu/dev.tsv', 'cv-corpus-27.0-2026-09-11/eu/invalidated.tsv', 'cv-corpus-27.0-2026-09-11/eu/other.tsv', 'cv-corpus-27.0-2026-09-11/eu/reported.tsv', 'cv-corpus-27.0-2026-09-11/eu/test.tsv', 'cv-corpus-27.0-2026-09-11/eu/train.tsv', 'cv-corpus-27.0-2026-09-11/eu/unvalidated_sentences.tsv', 'cv-corpus-27.0-2026-09-11/eu/validated.tsv', 'cv-corpus-27.0-2026-09-11/eu/validated_sentences.tsv', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251595.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251596.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251597.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251598.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251599.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice_eu_18251640.mp3', 'cv-corpus-27.0-2026-09-11/eu/clips/common_voice

In [7]:
import tarfile

with tarfile.open("basque-common-voice.tar.gz", "r:gz") as tar:
    tar.extract("cv-corpus-27.0-2026-09-11/eu/test.tsv", path=".")

print("test.tsv extracted")

/tmp/ipykernel_58/3993718690.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract("cv-corpus-27.0-2026-09-11/eu/test.tsv", path=".")


test.tsv extracted


In [8]:
import pandas as pd

test_df = pd.read_csv("cv-corpus-27.0-2026-09-11/eu/test.tsv", sep="\t")
print(test_df.shape)
print(test_df[["path", "sentence"]].head())

(14809, 13)
                           path  \
0  common_voice_eu_22343411.mp3   
1  common_voice_eu_39176289.mp3   
2  common_voice_eu_39176288.mp3   
3  common_voice_eu_39176290.mp3   
4  common_voice_eu_39176287.mp3   

                                            sentence  
0  Egun honek Latinoamerikan eta Karibean du jato...  
1  Egunaren amaieran txingorra egin lezake, eta t...  
2                  Emakumeak ihes egitea lortu zuen.  
3                  Ez da izan ikasi dudan gauza bat.  
4  Horrelako jarrerak mugatzeko neurriak hartu di...  


In [16]:
# TO DETERMINE COLUMNS IN TEST DATAFRAME
print(test_df.columns.tolist())

['client_id', 'path', 'sentence_id', 'sentence', 'sentence_domain', 'up_votes', 'down_votes', 'age', 'gender', 'accents', 'variant', 'locale', 'segment']


In [9]:
clip_names = set(test_df["path"])  # fast lookup
prefix = "cv-corpus-27.0-2026-09-11/eu/clips/"

with tarfile.open("basque-common-voice.tar.gz", "r:gz") as tar:
    for member in tar:
        if member.name.startswith(prefix) and member.name[len(prefix):] in clip_names:
            tar.extract(member, path=".")

print("Test clips extracted")

/tmp/ipykernel_58/3986926503.py:7: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=".")


Test clips extracted


In [10]:
import os
clips_path = "cv-corpus-27.0-2026-09-11/eu/clips"
print(len(os.listdir(clips_path)))

14809


### Authenticate with HuggingFace
Needed for higher rate limits / faster model downloads.

In [4]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("hf_basque_benchmarking")
login(token=hf_token)

## 3. Model: Whisper-large-v3
Load the model and build a transcription pipeline.

In [18]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if device == "cuda" else torch.float32

from transformers import pipeline
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

print("Model loaded")

asr_pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

Model loaded


[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


[' Egun honek Latinoamerikan eta Karibian du jatorria.', ' Egunaren amaieran txingorragin lezake, eta durmoiek aizak egoteko aukera ere egongo da.', ' Emakumeak iesegitea lortu zuen.', ' Ez deizan ikasi dudan gauzagat.', ' Horrelako jarrerak mugatzeko neurriak hartu ditugu.', ' Honek guztiak noski egoera larrian usten du klubaren etorkizuna bera ere.', ' Guk donostiako emakumeen etxeko erasoen kontrako bat zordetik.', ' Euskararen alde edo lanean ari diren elkarte guztiak federatzen ditugu.', ' Eta aldatzeko parada badago iraganean bai klase politikoaren, bai jende artearen.', ' Baina ez da, atxeten aldiaren osteko lehen lan saioak uziduen albistean bakarra.', ' Gaur egun merkatu handi dugun kontxola eta jokoekin, zergatik egiten duztuen denbora natera.', ' Dizentroetako langilei eta erabiltzailei prebentziosko behaketak egiten ari zaizkia.', ' Ira gazleriaren azken urteotako bilakaera ona izan da.', ' Atletak ez ditu galdera deselo zoenak enantzuna izan.', ' Goza ezazu Euskal Kulturaz

In [ ]:
import time
import librosa


def transcribe_with_timing(pipe, audio_path):
    duration = librosa.get_duration(path=audio_path)
    start = time.time()
    text = pipe(audio_path)["text"]
    elapsed = time.time() - start
    rtf = elapsed / duration
    return text, rtf

# Run inference + timing on the first 200 test clips (small sample to validate before scaling to all 14,801)
sample_paths = [f"{clips_path}/{name}" for name in test_df["path"].head(200)]
results = [transcribe_with_timing(asr_pipe, p) for p in sample_paths]
transcripts = [r[0] for r in results]
rtfs = [r[1] for r in results]

print(transcripts)
print(rtfs)

## 4. Metrics
WER, CER, and substitution/insertion/deletion breakdown, using jiwer.

In [21]:
import jiwer


def compute_wer(predictions, references):
    # normalize: lowercase, strip punctuation, so formatting differences don't count as errors
    transform = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.Strip(),
    ])
    predictions = [transform(p) for p in predictions]
    references = [transform(r) for r in references]
    
    return jiwer.wer(references, predictions)


def compute_cer(predictions, references):
    transform = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.Strip(),
    ])
    predictions = [transform(p) for p in predictions]
    references = [transform(r) for r in references]
    return jiwer.cer(references, predictions)


def compute_error_breakdown(predictions, references):
    transform = jiwer.Compose([
        jiwer.ToLowerCase(),
        jiwer.RemovePunctuation(),
        jiwer.Strip(),
    ])
    predictions = [transform(p) for p in predictions]
    references = [transform(r) for r in references]
    output = jiwer.process_words(references, predictions)
    return {
        "substitutions": output.substitutions,
        "insertions": output.insertions,
        "deletions": output.deletions,
    }

In [22]:
ground_truth = test_df["sentence"].head(200).tolist()

wer = compute_wer(transcripts, ground_truth)
cer = compute_cer(transcripts, ground_truth)
breakdown = compute_error_breakdown(transcripts, ground_truth)

print(f"WER: {wer}")
print(f"CER: {cer}")
print(breakdown)

WER: 0.35505896958410926
CER: 0.0646857524413655
{'substitutions': 445, 'insertions': 68, 'deletions': 59}
